# Cuaderno F · Lluvias e inundaciones

**LlajtaLab · NASA Space Apps 2026 Cochabamba · Enfoque 1**

Una inundación urbana se explica por **cuánto llueve, qué tan fuerte llueve, cuánto suelo está cubierto de cemento y qué tan bajo y plano es el terreno**. Este cuaderno mide todo eso, sus tendencias, y arma una **susceptibilidad a inundación** por distrito.

| Medida | Fuente | Periodo |
|---|---|---|
| Lluvia mensual por distrito | NASA GPM IMERG V07 | 2001–2025 |
| Lluvias fuertes y lluvia máxima en un día | NASA GPM IMERG V07 (cada 30 min) | 2001–2025 |
| Escorrentía (agua que corre por la superficie) | NASA GLDAS | 2001–2025 |
| Superficie construida | JRC GHSL | 2000–2025 |
| Terreno bajo y pendiente | NASA NASADEM y MERIT Hydro | Actual |
| Construcción sobre la cota 2750 (ladera del Tunari) | NASADEM + GHSL | 2000–2025 |
| Franja de 200 m del río Rocha y quiénes viven ahí | OpenStreetMap + GHSL 2025 | Actual |

**Sube a este Colab:**
1. `distritos_indicadores_ambientales_2024.geojson`
2. `tendencias_distritos.json` **el que salió del Cuaderno E** (el último, de `public/data/`)

Las celdas lentas guardan un CSV de respaldo y se saltan lo ya hecho si Colab se corta.

## Celda F1 · Preparación

In [1]:
!pip install -q earthengine-api pymannkendall
import ee, json, time, itertools, os, requests, pandas as pd, numpy as np, pymannkendall as mk, geopandas as gpd
from scipy import stats
from shapely.geometry import LineString
ee.Authenticate(); ee.Initialize(project='microclimas-cochabamba')

gj = json.load(open('/content/distritos_indicadores_ambientales_2024.geojson', encoding='utf-8'))
CAMPO_ID, CAMPO_NOMBRE = 'codigo_distrito', 'nombre'
distritos = ee.FeatureCollection(gj).select([CAMPO_ID, CAMPO_NOMBRE])
municipio = distritos.geometry().dissolve(100)
tj = json.load(open('/content/tendencias_distritos.json', encoding='utf-8'))
print('Variables que ya tiene el JSON:', list(tj['distritos'][0]['tendencias'].keys()))

def clave(v):
    return str(int(float(v)))

d_gdf = gpd.GeoDataFrame.from_features(gj['features'], crs=4326).to_crs(3857)
d_gdf['id'] = d_gdf[CAMPO_ID].map(clave)
vacia = ee.Image.constant(0).updateMask(ee.Image.constant(0))

def reintentar(fun, etiqueta):
    for intento in range(6):
        try:
            return fun()
        except Exception as e:
            print(etiqueta, 'reintento:', str(e)[:80]); time.sleep(20 * (intento + 1))
    raise RuntimeError(f'No se pudo: {etiqueta}')

def analizar_mensual(serie):
    s = serie.set_index('fecha').asfreq('MS'); v = s.iloc[:, 0].astype(float)
    r = mk.seasonal_test(v.values, period=12)
    return {'sen_pendiente_anual': round(float(r.slope), 4), 'mk_p': round(float(r.p), 4),
            'mk_tendencia': r.trend, 'significativa': bool(r.p < 0.05), 'meses_validos': int(v.notna().sum())}

def analizar_anual(anios, valores):
    s = pd.Series(valores, index=anios, dtype=float).dropna()
    if len(s) < 4:
        return {'sen_pendiente_anual': None, 'mk_p': None, 'mk_tendencia': 'datos insuficientes',
                'significativa': False, 'lineal_pendiente_anual': None, 'ic95': [None, None], 'anios_validos': int(len(s))}
    pares = [(s.iloc[j] - s.iloc[i]) / (s.index[j] - s.index[i]) for i, j in itertools.combinations(range(len(s)), 2)]
    r = mk.original_test(s.values)
    lr = stats.linregress(s.index.values.astype(float), s.values)
    tc = stats.t.ppf(0.975, len(s) - 2)
    return {'sen_pendiente_anual': round(float(np.median(pares)), 4), 'mk_p': round(float(r.p), 4),
            'mk_tendencia': r.trend, 'significativa': bool(r.p < 0.05),
            'lineal_pendiente_anual': round(float(lr.slope), 4),
            'ic95': [round(float(lr.slope - tc * lr.stderr), 4), round(float(lr.slope + tc * lr.stderr), 4)],
            'anios_validos': int(len(s))}

Variables que ya tiene el JSON: ['lst', 'ndvi', 'aod', 't2m', 'lst_noche', 'lst_landsat', 'lst_landsat_rel']


## Celda F2 · Lluvia mensual por distrito

**Revisa:** Cochabamba recibe unos cientos de milímetros al año, concentrados de diciembre a marzo. Si ves miles de mm o ceros, avisa.

In [2]:
ARCH_R = '/content/lluvia_mensual_distritos.csv'
imerg = ee.ImageCollection('NASA/GPM_L3/IMERG_V07').select('precipitation')          # mm/hora, cada 30 min
imerg_m = ee.ImageCollection('NASA/GPM_L3/IMERG_MONTHLY_V07').select('precipitation')  # mm/hora promedio del mes
USAR_MENSUAL = True
try:
    print('Imágenes mensuales IMERG disponibles:', imerg_m.filterDate('2001-01-01', '2002-01-01').size().getInfo())
except Exception as e:
    USAR_MENSUAL = False
    print('No está la colección mensual; se calcula desde la de 30 minutos (más lento).')

def lluvia_mes(y, m):
    ini = ee.Date.fromYMD(y, m, 1); fin = ini.advance(1, 'month')
    if USAR_MENSUAL:
        c = imerg_m.filterDate(ini, fin)
        horas = pd.Period(f'{y}-{m:02d}').days_in_month * 24
        return ee.Image(ee.Algorithms.If(c.size().gt(0), c.first(), vacia)).multiply(horas)
    return imerg.filterDate(ini, fin).sum().multiply(0.5)

if os.path.exists(ARCH_R):
    R = pd.read_csv(ARCH_R, dtype={'id': str}, parse_dates=['fecha']); print('Usando', ARCH_R)
else:
    filas = []
    for y in range(2001, 2026):
        img = ee.Image.cat([lluvia_mes(y, m).rename(f'p_{m:02d}') for m in range(1, 13)])
        res = reintentar(lambda: img.reduceRegions(collection=distritos, reducer=ee.Reducer.mean(), scale=1000, tileScale=4)
                         .map(lambda f: f.setGeometry(None)).getInfo()['features'], y)
        for f in res:
            for m in range(1, 13):
                filas.append({'id': clave(f['properties'][CAMPO_ID]), 'fecha': f'{y}-{m:02d}', 'lluvia': f['properties'].get(f'p_{m:02d}')})
        print('Listo', y)
    R = pd.DataFrame(filas); R['fecha'] = pd.to_datetime(R['fecha'])
    R.to_csv(ARCH_R, index=False)
print(R.groupby(R['fecha'].dt.year)['lluvia'].sum().div(R['id'].nunique()).round(0).rename('lluvia anual promedio (mm)'))

Imágenes mensuales IMERG disponibles: 12
Listo 2001
Listo 2002
Listo 2003
Listo 2004
Listo 2005
Listo 2006
Listo 2007
Listo 2008
Listo 2009
Listo 2010
Listo 2011
Listo 2012
Listo 2013
Listo 2014
Listo 2015
Listo 2016
Listo 2017
Listo 2018
Listo 2019
Listo 2020
Listo 2021
Listo 2022
Listo 2023
Listo 2024
Listo 2025
fecha
2001     933.0
2002     895.0
2003     971.0
2004     932.0
2005     842.0
2006     963.0
2007     902.0
2008    1012.0
2009     975.0
2010     948.0
2011    1039.0
2012    1006.0
2013     976.0
2014    1215.0
2015    1094.0
2016     944.0
2017    1023.0
2018    1207.0
2019     950.0
2020     937.0
2021     989.0
2022     822.0
2023     892.0
2024    1012.0
2025     848.0
Name: lluvia anual promedio (mm), dtype: float64


## Celda F3 · Lluvias fuertes en la ciudad

Suma la lluvia de cada día en toda la ciudad. Una "lluvia fuerte" es un día más lluvioso que el 95 % de los días con lluvia de 2001–2010.

Pide **un solo cálculo por mes** (todos sus días juntos), guarda después de cada año y se salta los años ya hechos.

In [3]:
ARCH_P = '/content/lluvia_diaria_ciudad.csv'

def lluvia_dias_mes(ini):
    dias = list(pd.date_range(ini, ini + pd.offsets.MonthEnd(0)))
    img = ee.Image.cat([imerg.filterDate(d.strftime('%Y-%m-%d'), (d + pd.Timedelta(days=1)).strftime('%Y-%m-%d'))
                        .sum().multiply(0.5).rename(d.strftime('d%Y%m%d')) for d in dias])
    vals = img.reduceRegion(ee.Reducer.mean(), municipio, 10000).getInfo()
    return [{'fecha': d, 'mm': vals.get(d.strftime('d%Y%m%d'))} for d in dias]

hechos = set(pd.read_csv(ARCH_P, parse_dates=['fecha'])['fecha'].dt.year) if os.path.exists(ARCH_P) else set()
for y in range(2001, 2026):
    if y in hechos:
        continue
    t0 = time.time(); filas = []
    for ini in pd.date_range(f'{y}-01-01', f'{y}-12-01', freq='MS'):
        filas += reintentar(lambda: lluvia_dias_mes(ini), ini.strftime('%Y-%m'))
    pd.DataFrame(filas).to_csv(ARCH_P, mode='a', header=not os.path.exists(ARCH_P), index=False)
    print('Listo', y, f'({time.time() - t0:.0f} s)')

P = pd.read_csv(ARCH_P, parse_dates=['fecha']).drop_duplicates('fecha'); P['anio'] = P['fecha'].dt.year
umbral = P[(P['anio'] <= 2010) & (P['mm'] >= 1)]['mm'].quantile(0.95)
LL = P.groupby('anio').agg(lluvia_total=('mm', 'sum'), lluvia_max_dia=('mm', 'max'),
                           dias_lluvia_fuerte=('mm', lambda s: int((s > umbral).sum())))
print('Umbral de lluvia fuerte:', round(umbral, 1), 'mm/día'); print(LL.round(1))

Listo 2001 (13 s)
Listo 2002 (13 s)
Listo 2003 (11 s)
Listo 2004 (13 s)
Listo 2005 (12 s)
Listo 2006 (10 s)
Listo 2007 (9 s)
Listo 2008 (9 s)
Listo 2009 (8 s)
Listo 2010 (9 s)
Listo 2011 (9 s)
Listo 2012 (10 s)
Listo 2013 (11 s)
Listo 2014 (9 s)
Listo 2015 (11 s)
Listo 2016 (9 s)
Listo 2017 (13 s)
Listo 2018 (11 s)
Listo 2019 (8 s)
Listo 2020 (9 s)
Listo 2021 (9 s)
Listo 2022 (8 s)
Listo 2023 (9 s)
Listo 2024 (7 s)
Listo 2025 (8 s)
Umbral de lluvia fuerte: 18.8 mm/día
      lluvia_total  lluvia_max_dia  dias_lluvia_fuerte
anio                                                  
2001         891.1            45.9                   6
2002         875.6            44.9                   6
2003         924.0            55.4                   9
2004         879.3            32.9                   5
2005         803.7            44.8                   8
2006         891.6            48.0                   7
2007         869.5            26.5                   5
2008         962.1            53

## Celda F4 · Escorrentía en la región del valle (GLDAS)

In [4]:
gldas = ee.ImageCollection('NASA/GLDAS/V021/NOAH/G025/T3H').select('Qs_acc')   # mm cada 3 horas
region_valle = municipio.buffer(15000)   # GLDAS tiene celdas de ~25 km

def escorrentia(y):
    img = ee.Image.cat([gldas.filterDate(ee.Date.fromYMD(y, m, 1), ee.Date.fromYMD(y, m, 1).advance(1, 'month'))
                        .sum().rename(f'e_{m:02d}') for m in range(1, 13)])
    vals = img.reduceRegion(ee.Reducer.mean(), region_valle, 25000).getInfo()
    return [vals.get(f'e_{m:02d}') for m in range(1, 13)]

filas = []
for y in range(2001, 2026):
    vals = reintentar(lambda: escorrentia(y), y)
    filas += [{'fecha': pd.Timestamp(y, m, 1), 'escorrentia': v} for m, v in zip(range(1, 13), vals)]
E = pd.DataFrame(filas)
print(E.groupby(E['fecha'].dt.year)['escorrentia'].sum().round(1))

fecha
2001     23.1
2002     21.9
2003     32.7
2004     21.2
2005     31.4
2006     35.6
2007     36.6
2008     41.9
2009     44.5
2010     45.4
2011     64.0
2012     58.1
2013     41.0
2014    101.2
2015     53.5
2016     74.9
2017    100.1
2018    136.1
2019     89.0
2020     34.7
2021     33.1
2022     17.5
2023     24.8
2024     28.3
2025     30.5
Name: escorrentia, dtype: float64


## Celda F5 · Superficie construida por distrito (2000–2025)

GHSL 2025 es una estimación del modelo; 2000 a 2020 son observados.

In [5]:
dist_area = distritos.map(lambda f: f.set('area_m2', f.geometry().area(10)))
filas = []
for y in [2000, 2005, 2010, 2015, 2020, 2025]:
    img = ee.Image(f'JRC/GHSL/P2023A/GHS_BUILT_S/{y}').select('built_surface')
    res = reintentar(lambda: img.reduceRegions(collection=dist_area, reducer=ee.Reducer.sum(), scale=100)
                     .map(lambda f: f.setGeometry(None)).getInfo()['features'], y)
    for f in res:
        p = f['properties']
        filas.append({'id': clave(p[CAMPO_ID]), 'anio': y, 'construido_pct': 100 * (p.get('sum') or 0) / p['area_m2']})
B = pd.DataFrame(filas)
print(B.pivot(index='id', columns='anio', values='construido_pct').round(1))

anio  2000  2005  2010  2015  2020  2025
id                                      
1     32.8  33.6  34.5  35.2  35.3  35.3
10    37.4  37.5  37.5  37.6  37.9  37.9
11    23.2  23.2  23.3  23.3  23.5  23.6
12    39.5  39.6  39.6  39.7  39.9  39.9
13     1.6   1.7   1.9   2.1   2.2   2.3
14    21.2  22.6  24.1  25.7  25.9  26.0
15     5.6   7.0   8.7  10.5  11.3  11.6
2     36.1  37.0  37.9  38.7  38.8  38.8
3     35.9  36.4  37.0  37.6  38.1  38.1
4     36.4  36.8  37.3  37.8  38.5  38.6
5     27.9  28.1  28.2  28.8  31.1  31.4
6     20.3  20.4  20.4  20.5  21.2  21.4
7     24.7  25.0  25.4  25.8  26.0  26.1
8     13.7  14.9  16.6  18.3  19.1  19.4
9      4.7   5.3   6.1   7.0   8.3   8.7


## Celda F6 · Terreno: bajo, plano y cercano al drenaje

HAND (altura sobre el drenaje más cercano): menos de 5 m es zona donde el agua tiende a acumularse.

In [6]:
dem = ee.Image('NASA/NASADEM_HGT/001').select('elevation')
terreno = ee.Image.cat([dem.rename('altitud'), ee.Terrain.slope(dem).rename('pendiente'),
                        ee.Image('MERIT/Hydro/v1_0_1').select('hnd').lt(5).rename('hand_bajo')])
res = reintentar(lambda: terreno.reduceRegions(collection=distritos, reducer=ee.Reducer.mean(), scale=90)
                 .map(lambda f: f.setGeometry(None)).getInfo()['features'], 'terreno')
TR = pd.DataFrame([{'id': clave(f['properties'][CAMPO_ID]), 'altitud': f['properties'].get('altitud'),
                    'pendiente': f['properties'].get('pendiente'), 'hand_bajo_pct': 100 * (f['properties'].get('hand_bajo') or 0)}
                   for f in res]).set_index('id')
print(TR.round(1))

    altitud  pendiente  hand_bajo_pct
id                                   
1    2673.1        6.1           17.8
10   2563.9        2.3           63.7
11   2622.6        9.8           24.2
12   2574.6        2.1           50.6
13   3502.5       20.0            6.8
14   2735.8       12.5            9.6
15   2810.4       10.5           12.3
2    2664.7        4.5           12.2
3    2561.4        1.7           64.2
4    2546.9        1.4           95.9
5    2551.0        1.7           79.5
6    2586.5        6.4           48.5
7    2633.0        9.0           30.0
8    2750.9       11.3           14.4
9    2724.1        8.0           32.9


## Celda F6b · Construcción sobre la cota 2750 (ladera del Tunari)

**Revisa:** solo los distritos del norte (los que suben hacia el Tunari) deberían tener valores mayores a cero.

In [7]:
alto = dem.gte(2750)
area_px = ee.Image.pixelArea()
filas = []
for y in [2000, 2005, 2010, 2015, 2020, 2025]:
    built = ee.Image(f'JRC/GHSL/P2023A/GHS_BUILT_S/{y}').select('built_surface')
    img = ee.Image.cat([built.updateMask(alto).rename('construido_ladera_m2'), area_px.updateMask(alto).rename('ladera_m2')])
    res = reintentar(lambda: img.reduceRegions(collection=distritos, reducer=ee.Reducer.sum(), scale=100)
                     .map(lambda f: f.setGeometry(None)).getInfo()['features'], y)
    for f in res:
        p = f['properties']
        filas.append({'id': clave(p[CAMPO_ID]), 'anio': y,
                      'construido_ladera_ha': (p.get('construido_ladera_m2') or 0) / 1e4,
                      'ladera_ha': (p.get('ladera_m2') or 0) / 1e4})
LAD = pd.DataFrame(filas)
print('Hectáreas construidas sobre los 2.750 m por distrito:')
print(LAD.pivot(index='id', columns='anio', values='construido_ladera_ha').round(1))
print('Total de la ciudad:', LAD.groupby('anio')['construido_ladera_ha'].sum().round(1).to_dict())

Hectáreas construidas sobre los 2.750 m por distrito:
anio  2000  2005  2010  2015  2020   2025
id                                       
1      0.9   0.9   0.9   0.9   0.9    0.9
10     0.0   0.0   0.0   0.0   0.0    0.0
11     0.1   0.1   0.1   0.1   0.1    0.1
12     0.0   0.0   0.0   0.0   0.0    0.0
13    69.9  77.9  87.0  95.0  98.2  102.9
14     3.6   4.5   5.6   6.7   7.7    8.3
15     8.5  14.4  23.4  34.7  45.6   50.0
2      3.5   3.6   3.7   3.8   3.8    3.8
3      0.0   0.0   0.0   0.0   0.0    0.0
4      0.0   0.0   0.0   0.0   0.0    0.0
5      0.0   0.0   0.0   0.0   0.0    0.0
6      0.0   0.0   0.0   0.0   0.0    0.0
7      0.2   0.4   0.6   1.3   1.7    1.7
8     19.4  27.2  40.6  53.5  63.6   67.7
9     15.5  25.9  40.2  57.8  80.5   88.3
Total de la ciudad: {2000: 121.5, 2005: 154.8, 2010: 202.1, 2015: 253.8, 2020: 302.1, 2025: 323.6}


## Celda F7 · El río Rocha y quiénes viven cerca

Si dice "0 tramos", avisa: buscamos el río con otro nombre.

In [8]:
q = '[out:json][timeout:90];way["waterway"="river"]["name"~"Rocha",i](-17.52,-66.27,-17.26,-66.07);out geom;'
resp = None
for url in ['https://overpass-api.de/api/interpreter', 'https://overpass.kumi.systems/api/interpreter']:
    try:
        resp = requests.post(url, data={'data': q}, timeout=120).json(); break
    except Exception as e:
        print('Falló', url, str(e)[:60])
rio = gpd.GeoDataFrame(geometry=[LineString([(p['lon'], p['lat']) for p in w['geometry']])
                                 for w in resp['elements'] if 'geometry' in w], crs=4326)
print(len(rio), 'tramos del río Rocha')
buf = rio.to_crs(3857).buffer(200 / np.cos(np.radians(17.4)))     # 200 m reales a cada lado
franja = buf.union_all() if hasattr(buf, 'union_all') else buf.unary_union

d_gdf['franja_pct'] = (d_gdf.geometry.intersection(franja).area / d_gdf.area * 100).round(2)
franja_ee = ee.Geometry(gpd.GeoSeries([franja], crs=3857).to_crs(4326).simplify(0.0001).iloc[0].__geo_interface__)
pob = ee.Image('JRC/GHSL/P2023A/GHS_POP/2025').select('population_count').clip(franja_ee)
res = reintentar(lambda: pob.reduceRegions(collection=distritos, reducer=ee.Reducer.sum(), scale=100)
                 .map(lambda f: f.setGeometry(None)).getInfo()['features'], 'poblacion franja')
d_gdf['personas_franja_rocha'] = d_gdf['id'].map({clave(f['properties'][CAMPO_ID]): round(f['properties'].get('sum') or 0) for f in res}).fillna(0)
print(d_gdf[['id', 'franja_pct', 'personas_franja_rocha']])
print('Total de personas a menos de 200 m del río Rocha:', int(d_gdf['personas_franja_rocha'].sum()))

Falló https://overpass-api.de/api/interpreter Expecting value: line 1 column 1 (char 0)
21 tramos del río Rocha
    id  franja_pct  personas_franja_rocha
0    1        2.80                   1050
1   10       14.63                   4599
2   11        9.85                   2295
3   12       10.65                   5624
4   13        0.00                      0
5   14        0.00                      0
6   15        0.00                      0
7    2        0.00                      0
8    3        0.00                      0
9    4        6.00                   2465
10   5        5.27                    681
11   6        0.00                      0
12   7        0.00                      0
13   8        0.00                      0
14   9        1.62                    259
Total de personas a menos de 200 m del río Rocha: 16973


## Celda F8 · Susceptibilidad a inundación por distrito

Seis componentes de 0 a 100 comparando a los 15 distritos (100 = el más susceptible). El promedio se clasifica en Baja, Media o Alta (tercios).

In [9]:
S = TR.join(d_gdf.set_index('id')[['franja_pct', 'personas_franja_rocha']])
S['construido_2020'] = B[B['anio'] == 2020].set_index('id')['construido_pct']
S['cambio_construido'] = S['construido_2020'] - B[B['anio'] == 2000].set_index('id')['construido_pct']
lad = LAD.pivot(index='id', columns='anio', values='construido_ladera_ha')
S['construido_ladera_ha'] = lad[2020]
S['cambio_ladera_ha'] = lad[2020] - lad[2000]

COMP_INUND = {   # nombre visible: (columna, ¿más alto = más susceptible?)
    'Terreno bajo respecto al drenaje': ('hand_bajo_pct', True),
    'Terreno plano': ('pendiente', False),
    'Suelo construido': ('construido_2020', True),
    'Aumento de construcción 2000–2020': ('cambio_construido', True),
    'Cercanía al río Rocha': ('franja_pct', True),
    'Construcción en la ladera del Tunari (sobre la cota 2750)': ('construido_ladera_ha', True)}
def norm(s, alto):
    s = s.astype(float)
    if s.max() == s.min():
        return s * 0
    return (s - s.min()) / (s.max() - s.min()) * 100 if alto else (s.max() - s) / (s.max() - s.min()) * 100
PI = pd.DataFrame({k: norm(S[c], a) for k, (c, a) in COMP_INUND.items()})
S['susceptibilidad'] = PI.mean(axis=1).round(1)
S['nivel'] = pd.qcut(S['susceptibilidad'].rank(method='first'), 3, labels=['Baja', 'Media', 'Alta']).astype(str)
print(S[['susceptibilidad', 'nivel', 'construido_2020', 'cambio_construido', 'hand_bajo_pct', 'pendiente',
         'franja_pct', 'personas_franja_rocha', 'construido_ladera_ha']].sort_values('susceptibilidad', ascending=False).round(1))

    susceptibilidad  nivel  construido_2020  cambio_construido  hand_bajo_pct  \
id                                                                              
4              62.1   Alta             38.5                2.2           95.9   
10             59.7   Alta             37.9                0.5           63.7   
5              57.6   Alta             31.1                3.2           79.5   
12             53.3   Alta             39.9                0.4           50.6   
3              49.0   Alta             38.1                2.2           64.2   
9              43.9  Media              8.3                3.6           32.9   
8              43.2  Media             19.1                5.4           14.4   
1              39.5  Media             35.3                2.6           17.8   
2              39.1  Media             38.8                2.7           12.2   
15             37.9  Media             11.3                5.7           12.3   
11             33.0   Baja  

## Celda F9 · Exportar

Descarga `lluvias_inundaciones.json` y `tendencias_distritos.json` a `public/data/`, y los CSV (`lluvia_mensual_distritos.csv`, `lluvia_diaria_ciudad.csv`) a `data/tendencias_modis/`. Guarda el cuaderno como `notebooks/lluvias_inundaciones.ipynb`.

In [10]:
num = lambda v: None if v is None or pd.isna(v) else round(float(v), 2)
nombres = {clave(d['id']): d['nombre'] for d in tj['distritos']}

# 1) Archivo propio de la pestaña Lluvias e inundaciones
salida = {'meta': {
    'fuentes': ['NASA GPM IMERG V07 (lluvia)', 'NASA GLDAS 2.1 (escorrentía)', 'NASA NASADEM (altitud y pendiente)',
                'MERIT Hydro (altura sobre el drenaje)', 'JRC GHSL (superficie construida y población)', 'OpenStreetMap (río Rocha)'],
    'umbral_lluvia_fuerte_mm': round(float(umbral), 1),
    'componentes_susceptibilidad': list(COMP_INUND.keys()),
    'nota': 'La susceptibilidad compara a los 15 distritos según terreno, construcción y cercanía al río. No es un mapa oficial de riesgo ni predice inundaciones puntuales.',
    'nota_ladera': 'Construcción sobre los 2.750 m de altitud (NASADEM), límite de urbanización del Parque Tunari según la cota 2750 del GAMC. GHSL 2025 es estimado.'},
  'ciudad': {'anual': [{'anio': int(a), **{k: num(v) for k, v in r.items()}} for a, r in LL.iterrows()],
             'tendencias': {c: analizar_anual(LL.index.values, LL[c].values) for c in ['lluvia_total', 'dias_lluvia_fuerte', 'lluvia_max_dia']},
             'escorrentia': {'serie': [{'fecha': f.strftime('%Y-%m'), 'mm': num(v)} for f, v in zip(E['fecha'], E['escorrentia'])],
                             'tendencia': analizar_mensual(E[['fecha', 'escorrentia']])}},
  'rio_rocha': json.loads(rio.simplify(0.0002).to_json()),
  'distritos': []}
for did, r in S.iterrows():
    b = B[B['id'] == did].sort_values('anio')
    salida['distritos'].append({'id': did, 'nombre': nombres.get(did, did),
        'susceptibilidad': {'puntaje': num(r['susceptibilidad']), 'nivel': r['nivel'],
                            'componentes': {k: {'valor': num(r[c]), 'puntaje': num(PI.loc[did, k])} for k, (c, _) in COMP_INUND.items()}},
        'construido': [{'anio': int(x.anio), 'pct': num(x.construido_pct)} for x in b.itertuples()],
        'altitud_m': num(r['altitud']), 'personas_franja_rocha': int(r['personas_franja_rocha'] or 0),
        'ladera': {'construido_ha': [{'anio': int(x.anio), 'ha': num(x.construido_ladera_ha)} for x in LAD[LAD['id'] == did].sort_values('anio').itertuples()],
                   'cambio_2000_2020_ha': num(r['cambio_ladera_ha'])}})
salida['distritos'].sort(key=lambda d: int(d['id']))
tot = LAD.groupby('anio')['construido_ladera_ha'].sum()
salida['ciudad']['ladera_total'] = {'serie': [{'anio': int(a), 'ha': num(v)} for a, v in tot.items()],
                                    'tendencia': analizar_anual(tot.index.values, tot.values)}
json.dump(salida, open('lluvias_inundaciones.json', 'w', encoding='utf-8'), ensure_ascii=False)

# 2) Agregar lluvia y construcción a tendencias_distritos.json (para el Detective de tendencias)
for d in tj['distritos']:
    did = clave(d['id'])
    sub = R[R['id'] == did]
    m = pd.Series(sub['lluvia'].values, index=sub['fecha'].dt.strftime('%Y-%m'))
    for p in d['serie']:
        p['lluvia'] = num(m.get(p['fecha']))
    d['tendencias']['lluvia'] = analizar_mensual(sub[['fecha', 'lluvia']])
    anual = {a['anio']: a for a in d.get('anual', [])}
    b = B[B['id'] == did]
    for x in b.itertuples():
        anual.setdefault(int(x.anio), {'anio': int(x.anio)})['construido_pct'] = num(x.construido_pct)
    d['anual'] = sorted(anual.values(), key=lambda a: a['anio'])
    d['tendencias']['construido_pct'] = analizar_anual(b['anio'].values, b['construido_pct'].values)
ciudad = tj.setdefault('ciudad', {'anual': [], 'tendencias': {}})
ca = {a['anio']: a for a in ciudad.get('anual', [])}
for a, r in LL.iterrows():
    ca.setdefault(int(a), {'anio': int(a)}).update({k: num(v) for k, v in r.items()})
ciudad['anual'] = sorted(ca.values(), key=lambda a: a['anio'])
ciudad.setdefault('tendencias', {}).update(salida['ciudad']['tendencias'])
tj['meta'].update({'fuente_lluvia': 'NASA GPM IMERG V07', 'periodo_lluvia': '2001-01 a 2025-12',
                   'nota_lluvia': 'La lluvia satelital tiene una resolución de unos 10 km: distritos vecinos pueden mostrar valores parecidos.',
                   'fuente_construido': 'JRC GHSL P2023A (superficie construida, 100 m)'})
tj['meta'].setdefault('unidades', {}).update({'lluvia': 'mm/año', 'lluvia_total': 'mm/año', 'dias_lluvia_fuerte': 'días/año',
                                              'lluvia_max_dia': 'mm/año', 'construido_pct': 'puntos porcentuales/año'})
json.dump(tj, open('tendencias_distritos.json', 'w', encoding='utf-8'), ensure_ascii=False)
print('Tendencias de la ciudad:')
for k, v in salida['ciudad']['tendencias'].items():
    print(f"  {k}: {v['sen_pendiente_anual']} por año, p = {v['mk_p']} ({v['mk_tendencia']})")
print('  escorrentía:', salida['ciudad']['escorrentia']['tendencia'])
print('  construcción sobre la cota 2750:', salida['ciudad']['ladera_total']['tendencia'])

Tendencias de la ciudad:
  lluvia_total: 3.2336 por año, p = 0.129 (no trend)
  dias_lluvia_fuerte: 0.0 por año, p = 0.7404 (no trend)
  lluvia_max_dia: -0.1892 por año, p = 0.3153 (no trend)
  escorrentía: {'sen_pendiente_anual': 0.0074, 'mk_p': 0.0144, 'mk_tendencia': 'increasing', 'significativa': True, 'meses_validos': 300}
  construcción sobre la cota 2750: {'sen_pendiente_anual': 8.8175, 'mk_p': 0.0085, 'mk_tendencia': 'increasing', 'significativa': True, 'lineal_pendiente_anual': 8.595, 'ic95': [7.3879, 9.8021], 'anios_validos': 6}
